In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install librosa opensmile transformers torch nltk soundfile

import os
import librosa
import numpy as np
import pandas as pd
import opensmile
import torch
import soundfile as sf
from transformers import BertTokenizer, BertModel
import nltk

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 33.9 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [3]:
# BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()

# OpenSMILE (includes jitter/shimmer)
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.ComParE_2016,
    feature_level=opensmile.FeatureLevel.Functionals,
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def extract_all_features(audio_path, text):

    y, sr = librosa.load(audio_path, sr=16000)

    # handle empty audio
    if len(y) == 0:
        return None

    # =========================
    # ACOUSTIC FEATURES
    # =========================
    n_fft = min(2048, len(y))

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=80, n_fft=n_fft
    )
    mel_db = librosa.power_to_db(mel)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    pitch, _ = librosa.piptrack(y=y, sr=sr)
    pitch = pitch[pitch > 0]
    pitch_mean = np.mean(pitch) if len(pitch) > 0 else 0

    energy = np.mean(librosa.feature.rms(y=y))

    # =========================
    # BEHAVIORAL FEATURES
    # =========================
    intervals = librosa.effects.split(y, top_db=20)

    pauses = []
    for i in range(len(intervals) - 1):
        end_prev = intervals[i][1]
        start_next = intervals[i+1][0]
        pause_duration = (start_next - end_prev) / sr

        if pause_duration > 0.2:
            pauses.append(pause_duration)

    pause_count = len(pauses)
    avg_pause = np.mean(pauses) if len(pauses) > 0 else 0

    words = text.split()
    duration = librosa.get_duration(y=y, sr=sr)

    wpm = len(words) / duration * 60 if duration > 0 else 0
    filler_count = text.lower().count("uh") + text.lower().count("um")
    repetition = sum(1 for i in range(len(words)-1) if words[i] == words[i+1])
    turn_length = len(words)

    # =========================
    # OPENSMILE (SAFE)
    # =========================
    try:
        smile_features = smile.process_file(audio_path).values.flatten()
        smile_features = np.nan_to_num(smile_features)
    except:
        smile_features = np.zeros(6373)

    # =========================
    # TEXT FEATURES
    # =========================
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

        with torch.no_grad():
            outputs = bert_model(**inputs)

        bert_embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    except:
        bert_embedding = np.zeros(768)

    # Tokenization + POS
    try:
        tokens = nltk.word_tokenize(text)
    except:
        tokens = text.split()

    lexical_diversity = len(set(tokens)) / len(tokens) if len(tokens) > 0 else 0
    sentence_length = len(tokens)

    try:
        pos_tags = nltk.pos_tag(tokens)
        pos_counts = {}

        for _, tag in pos_tags:
            pos_counts[tag] = pos_counts.get(tag, 0) + 1

        total_tags = len(pos_tags)
        pos_distribution = {
            k: v / total_tags for k, v in pos_counts.items()
        } if total_tags > 0 else {}
    except:
        pos_distribution = {}

    return {
        # Acoustic
        "mel": mel_db,
        "mfcc": mfcc,
        "pitch": pitch_mean,
        "energy": energy,

        # Behavioral
        "pause_count": pause_count,
        "avg_pause": avg_pause,
        "speech_rate": wpm,
        "filler_count": filler_count,
        "repetition": repetition,
        "turn_length": turn_length,

        # Text
        "lexical_diversity": lexical_diversity,
        "sentence_length": sentence_length,
        "pos_distribution": pos_distribution,

        # Embeddings
        "bert": bert_embedding,
        "smile": smile_features
    }

In [6]:
data_csv = "/content/drive/MyDrive/Segmented_Data/final_segments.csv"
df = pd.read_csv(data_csv)

features = []

for i, row in df.iterrows():

    audio_path = row["audio"]
    text = row["text"]

    print(f"Processing {i}...")

    try:
        y, sr = librosa.load(audio_path, sr=16000)
        duration = librosa.get_duration(y=y, sr=sr)

        # optional safety (you can remove if using merged segments)
        if duration < 0.2:
            print(f"Skipping very tiny segment {i}")
            continue

        feat = extract_all_features(audio_path, text)

        if feat is None:
            continue

        features.append({
            "audio": audio_path,
            "text": text,
            "features": feat,
            "label": 1 if "dementia" in audio_path else 0
        })

    except Exception as e:
        print(f"Error at {i}: {e}")

Processing 0...
Processing 1...
Processing 2...
Processing 3...
Processing 4...
Processing 5...
Processing 6...
Processing 7...
Processing 8...
Processing 9...
Processing 10...
Processing 11...
Processing 12...
Processing 13...
Processing 14...
Processing 15...
Processing 16...
Processing 17...
Processing 18...
Processing 19...
Processing 20...
Processing 21...
Processing 22...
Processing 23...
Skipping very tiny segment 23
Processing 24...
Processing 25...
Processing 26...
Processing 27...
Processing 28...
Processing 29...
Processing 30...
Processing 31...
Processing 32...
Processing 33...
Processing 34...
Processing 35...
Processing 36...
Processing 37...
Processing 38...
Processing 39...
Processing 40...
Processing 41...
Processing 42...
Processing 43...
Processing 44...
Processing 45...
Processing 46...
Processing 47...
Processing 48...
Processing 49...
Processing 50...
Processing 51...
Processing 52...
Processing 53...
Processing 54...
Processing 55...
Processing 56...
Processing 

In [7]:
import pickle

save_path = "/content/drive/MyDrive/final_features.pkl"

with open(save_path, "wb") as f:
    pickle.dump(features, f)

print("✅ FINAL FEATURES SAVED")

✅ FINAL FEATURES SAVED
